# 🎬 Geração de Vídeo Cinematográfico a partir de Imagens

In [ ]:

!pip install diffusers accelerate einops transformers ffmpeg-python gradio --quiet


In [ ]:

import os
import torch
import ffmpeg
from PIL import Image
import gradio as gr
from diffusers import StableVideoDiffusionPipeline

os.makedirs("inputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)


In [ ]:

def generate_video(image_path, style_prompt, duration=2, fps=14):
    model_id = "stabilityai/stable-video-diffusion-img2vid-xt"
    pipe = StableVideoDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
    pipe = pipe.to("cuda")

    image = Image.open(image_path).convert("RGB")
    image = image.resize((576, 1024))

    output = pipe(prompt=style_prompt, image=image, decode_chunk_size=8, num_frames=25)
    frames = output.frames
    output_path = os.path.join("outputs", os.path.basename(image_path).split(".")[0] + ".mp4")
    frames[0].save(output_path, save_all=True, append_images=frames[1:], duration=int(1000/fps), loop=0)

    return output_path


In [ ]:

def video_app(img, style, duration, fps):
    input_path = os.path.join("inputs", img.name)
    with open(input_path, "wb") as f:
        f.write(img.read())

    result = generate_video(input_path, style, duration, fps)
    return result


In [ ]:

with gr.Blocks() as demo:
    gr.Markdown("## 🎥 Geração de Vídeos Cinematográficos a partir de Imagens")
    with gr.Row():
        img_input = gr.Image(label="Imagem de Entrada", type="file")
        style_input = gr.Textbox(label="Prompt de Estilo (ex: 'anime watercolor, cinematic')", value="cinematic")
    with gr.Row():
        duration_input = gr.Slider(label="Duração por Clipe (s)", minimum=1, maximum=10, step=0.5, value=2)
        fps_input = gr.Slider(label="FPS", minimum=6, maximum=30, step=1, value=14)
    generate_btn = gr.Button("Gerar Vídeo")
    video_output = gr.Video(label="Resultado")

    generate_btn.click(fn=video_app, inputs=[img_input, style_input, duration_input, fps_input], outputs=video_output)

demo.launch(share=True)
